# **Problem Statement**

## **Business Context**

ShopNest Global is a large-scale e-commerce platform operating across 30+ countries, serving over 50 million active customers and employing 2,000+ human support agents who run 24/7 across the US, Europe, India, and Southeast Asia, across product categories including electronics, fashion, groceries, and home appliances.

ShopNest processes over 200,000 orders per day. With every order comes the possibility of a delivery delay, a payment failure, a wrong item, or a return request. Customers reach out to the support team through tickets to report these issues and expect a fast, accurate resolution. These tickets are written in highly unstructured ways:

- Some are overloaded with background details where the real issue is buried.
- Others use abbreviations, shorthand, and order codes that are hard to interpret.
- Some contain so little information that the issue is entirely unclear.

As a result:

- Human agents spend the first **2–3 minutes** on each ticket just decoding what the customer is asking before any resolution work can begin.
- During peak periods (sales, holidays, logistics disruptions), daily ticket volume can spike from ~5,000 to **~15,000**, multiplying this inefficiency.
- High volume and inconsistent ticket content contribute to agent fatigue and higher error rates when accuracy is most critical.
- Drafting responses is manual, slow, and inconsistent in tone and clarity across agents, leading to suboptimal customer experiences.

## **Objective**

The objective is to build a POC of an AI-powered ticket intelligence system for ShopNest Global that:

1. **Summarises** incoming raw, unstructured tickets into a clean, concise summary for the support agent.
2. **Evaluates** the generated summary using an LLM-as-Judge approach, scoring quality on defined criteria.
3. **Generates** a professional, empathetic customer response grounded in ShopNest's support policies.
4. **Evaluates** the generated response using an LLM-as-Judge approach, scoring resolution quality.
5. **Compiles** all outputs into a single structured table and exports it for downstream use.

The end goal is to demonstrate that AI-assisted summarisation and response generation can meaningfully improve the consistency and quality of customer support operations at scale.

## **Data Dictionary**

| Column Name         | Data Type | Description                                                       |
| ------------------- | --------- | ----------------------------------------------------------------- |
| support_ticket_id   | Integer   | Unique identifier assigned to each support ticket                 |
| support_ticket_text | String    | Free-form text describing the issue or request raised by the user |

# **Installing and Importing Necessary Libraries**

In [1]:
# Install LangChain and OpenAI for LLM API access, and pandas for data handling
# Pinned versions ensure reproducibility across environments
%pip install pandas==2.2.2 langchain-openai==1.1.12 openai==2.31.0

  Obtaining dependency information for pandas==2.2.2 from https://files.pythonhosted.org/packages/22/a5/a0b255295406ed54269814bc93723cfd1a0da63fb9aaf99e1364f07923e5/pandas-2.2.2-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for langchain-openai==1.1.12 from https://files.pythonhosted.org/packages/6e/a6/68fb22e3604015e6f546fa1d3677d24378b482855ae74710cbf4aec44132/langchain_openai-1.1.12-py3-none-any.whl.metadata
  Obtaining dependency information for openai==2.31.0 from https://files.pythonhosted.org/packages/66/bc/a8f7c3aa03452fedbb9af8be83e959adba96a6b4a35e416faffcc959c568/openai-2.31.0-py3-none-any.whl.metadata
  Obtaining dependency information for numpy>=1.26.0 from https://files.pythonhosted.org/packages/65/66/53f31807a48a750f9d748da273bc3fcedd12b27ff1f3e373bfec55ef2dc0/numpy-2.5.1-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for pytz>=2020.1 from https://files.pythonhosted.org/packages/0f/7b/39c34ca613b0b198cb866466651b26b045e20098


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [78]:
# Core libraries
import os            # For environment variable access (API key fallback)
import json
import re            # For parsing structured evaluator output into numeric scores
import pandas as pd  # For loading, manipulating, and exporting tabular data
from langchain_openai import ChatOpenAI  # OpenAI
from openai import OpenAI

## **Loading the Open API Key**

In [113]:
# Load the JSON file and extract values
file_name = 'C:\\datascience\\SupportTicketAnalysis\\config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")
# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                  # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable
client = OpenAI()

# **Data Loading**

## **Load the data**

In [114]:
file_path = r"C:\datascience\SupportTicketAnalysis\support_ticket_data.csv"
df = pd.read_csv(file_path)

In [115]:
df.shape

(30, 2)

## **Data Overview**

In [117]:
print(df.head())

   support_ticket_id                                support_ticket_desc
0                  1  I cannot believe the level of service I have r...
1                  2  Ord SNX-8902 ACH debit failed at checkout, tri...
2                  3                          not working. please help.
3                  4  Okay so I have genuinely had it with this comp...
4                  5                                refund not received


In [118]:
print(df.tail())

    support_ticket_id                                support_ticket_desc
25                 26  Hi, just wanted to check on the status of my o...
26                 27  Hello, quick question - what is your standard ...
27                 28  Hi there. I received my order SNX-7823 today a...
28                 29  Hey, I placed an order yesterday evening - ord...
29                 30  Hi, I was just checking whether my recent retu...


# **Summarization**

## **Set up an LLM**

In [ ]:
MODEL_NAME = "gpt-4o-mini"

## **Set parameters**

In [146]:
SUMMARY_TEMP = 0.1
SUMMARY_MAX_TOKENS = 50
SUMMARY_TOP_P = 0.95

## **Prompting Technique**

### **System Message**

In [158]:
SUMMARISER_SYSTEM = """
You are an assistant that converts raw customer support tickets into a concise, agent-ready summary.
Extract the main issue, relevant details, and the customer’s request.
Keep the summary short, factual, and free of extra explanation.
Do not add anything that is not present in the ticket.
"""

In [150]:
SUMMARISER_USER = """
Read the ticket below and write a brief support summary in one or two sentences.
Ticket: "{ticket}"
Summary:
"""

### **Generate Ticket Summaries**

In [152]:
def generate_summary(ticket: str) -> str:
    user_message = SUMMARISER_USER.format(ticket=ticket)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=SUMMARY_TEMP,
        max_tokens=SUMMARY_MAX_TOKENS,
        top_p=SUMMARY_TOP_P,
        messages=[
            {"role": "system", "content": SUMMARISER_SYSTEM},
            {"role": "user", "content": user_message}
        ]
    )

    return response.choices[0].message.content.strip()

In [159]:
test_ticket = df["support_ticket_desc"].iloc[0]

print("INPUT TICKET:")
print("-" * 60)
print(test_ticket)

print("\nGENERATED SUMMARY:")
print("-" * 60)
test_summary = generate_summary(test_ticket)
print(test_summary)

INPUT TICKET:
------------------------------------------------------------
I cannot believe the level of service I have received. I have been a loyal ShopNest customer since 2020 and have spent thousands of dollars on this platform. I have recommended ShopNest to everyone I know - my coworkers, my neighbors, my entire book club. I have never once filed a complaint or asked for anything special. I always leave five star reviews and I tip delivery drivers generously. And THIS is how you treat your most loyal customers? I am beyond frustrated. I spent over three hours today trying to reach your support team and not a single person gave me a straight answer. Your chatbot is completely useless, your email support is nonexistent, and your phone line had a 40 minute hold time. I have screenshots of every single interaction. I will be filing a complaint with the Better Business Bureau if this is not resolved by end of day. Anyway. Order SNX-4421 for a Bosch dishwasher. Delivered wrong model. W

### **Observations**

Observed the Change in temprature add more context other than mentioned in the ticket

Observed the Max token changes

# **Evaluation for Summarization**

## **Setup an LLM**

In [154]:
MODEL_NAME = "gpt-4o"

## **System Message**

In [160]:
SUMMARY_EVAL_SYSTEM = """
You are a JSON evaluator for customer support ticket summaries.
Compare the summary against the original ticket and return ONLY valid JSON.
Include keys: scores, overall_verdict.
The scores object must contain technical_accuracy, completeness, conciseness, hallucination_check.
"""

In [ ]:
SUMMARY_EVAL_USER = """\
Original Ticket: "{ticket}"
Ticket Summary: "{summary}"
Evaluate the summary and respond ONLY with valid JSON like:
{{
  "scores": {{
    "technical_accuracy": 1,
    "completeness": 1,
    "conciseness": 1,
    "hallucination_check": 1
  }},
  "overall_verdict": "..."
}}
"""

## **Generate Evaluation Scores**

In [156]:
# Write your code here
def evaluate_summary(ticket: str, summary: str) -> str:
    # Escape any literal braces in ticket or summary text before formatting the prompt
    safe_ticket = ticket.replace("{", "{{").replace("}", "}}")
    safe_summary = summary.replace("{", "{{").replace("}", "}}")
    user_message = SUMMARY_EVAL_USER.format(ticket=safe_ticket, summary=safe_summary)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SUMMARY_EVAL_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [157]:
print("SUMMARISATION EVALUATION")
print("=" * 60)

test_sum_eval_raw = evaluate_summary(test_ticket, test_summary)
print(test_sum_eval_raw)

SUMMARISATION EVALUATION
{
  "scores": {
    "technical_accuracy": 1,
    "completeness": 0.75,
    "conciseness": 1,
    "hallucination_check": 1
  },
  "overall_verdict": "The summary is technically accurate and concise, but it lacks completeness as it omits the customer's intent to file a complaint with the Better Business Bureau and details about their long-term loyalty and efforts in promoting the brand."
}


## **Observations**

Observed the scores for various model with different Parameters

# **Response Generation**

## **Set up an LLM**

In [ ]:
MODEL_NAME = "gpt-4o"

## **Set parameters**

In [ ]:
GENERATION_TEMP = 0.5

## **Prompting Technique**

### **System Message**

In [169]:
GENERATOR_SYSTEM = """
You are a customer support assistant. Use the ticket summary to write a polite, helpful reply.
Respond directly to the issue, offer a clear next step or resolution, and keep the tone professional and empathetic.
Do not add any information that is not supported by the summary.
"""

In [170]:
GENERATOR_USER = """
Use the summary below to write a customer-facing response.
Keep the reply concise, courteous, and focused on resolving the ticket.
Ticket Summary: "{summary}"
"""

### **Generate a User Response**

In [166]:
def generate_response(summary: str) -> str:
    # Inject the summary into the CoT user message template
    user_message = GENERATOR_USER.format(summary=summary)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=GENERATION_TEMP,       # Moderate temperature for natural, empathetic tone
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [167]:
print("INPUT SUMMARY:")
print("-" * 60)
print(test_summary)

print("\nGENERATED RESPONSE:")
print("-" * 60)
test_response = generate_response(test_summary)
print(test_response)

INPUT SUMMARY:
------------------------------------------------------------
Order SNX-4421 for a Bosch dishwasher was delivered with the wrong model; the customer requests a replacement and expresses frustration with support response times.

GENERATED RESPONSE:
------------------------------------------------------------
Dear [Customer's Name],

Thank you for reaching out to us and bringing this to our attention. I sincerely apologize for the inconvenience caused by receiving the incorrect Bosch dishwasher model and for any delays in our response.

To resolve this issue promptly, I have initiated the process for a replacement with the correct model. Our team will contact you shortly to arrange for the return of the incorrect item and schedule the delivery of the correct dishwasher.

We appreciate your patience and understanding. Please feel free to reach out if you have any further questions or need additional assistance.

Warm regards,

[Your Name]  
[Your Position]  
[Company Name]  

In [168]:
# Write your code here

### **Observations**

Write your observations here

# **Evaluation for Response Generation**

## **Setup an LLM**

In [ ]:
MODEL_NAME = "gpt-4o"

## **System Message**

In [ ]:
RESPONSE_EVAL_SYSTEM = """
You are a JSON evaluator for customer support responses.
Compare the generated response against the ticket summary and return ONLY valid JSON.
Include keys: scores, feedback, overall_verdict.
The scores object must contain alignment_with_summary, actionability, tone_empathy, policy_compliance.
The feedback object must contain strengths, weaknesses, risk_factors.
"""

In [ ]:
RESPONSE_EVAL_USER = """\
Ticket Summary: "{summary}"
Generated Response: "{response}"

Evaluate the response and respond ONLY with valid JSON like:
{{
  "scores": {{
    "alignment_with_summary": 1,
    "actionability": 1,
    "tone_empathy": 1,
    "policy_compliance": 1
  }},
  "feedback": {{
    "strengths": "...",
    "weaknesses": "...",
    "risk_factors": "..."
  }},
  "overall_verdict": "..."
}}

"""

## **Generate Evaluation Scores**

In [171]:
def evaluate_response(summary: str, response_text: str) -> str:
    # Escape any literal braces in summary or response text before formatting the prompt
    safe_summary = summary.replace("{", "{{").replace("}", "}}")
    safe_response = response_text.replace("{", "{{").replace("}", "}}")
    user_message = RESPONSE_EVAL_USER.format(summary=safe_summary, response=safe_response)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": RESPONSE_EVAL_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [172]:
print("RESPONSE EVALUATION")
print("=" * 60)

test_resp_eval_raw = evaluate_response(test_summary, test_response)
print(test_resp_eval_raw)

RESPONSE EVALUATION
```json
{
  "scores": {
    "alignment_with_summary": 1,
    "actionability": 1,
    "tone_empathy": 1,
    "policy_compliance": 1
  },
  "feedback": {
    "strengths": "The response fully addresses the issue of receiving the wrong model and initiates a replacement process, demonstrating clear actionability. The tone of the message is empathetic and acknowledges the customer's frustration, aligning well with the ticket summary.",
    "weaknesses": "The response could include an estimated timeline for the replacement process to provide more clarity to the customer.",
    "risk_factors": "If there is any delay in the replacement process or in contacting the customer, it could exacerbate the customer's frustration."
  },
  "overall_verdict": "The response effectively aligns with the ticket summary, is actionable, empathetic, and adheres to policy. It addresses the customer's frustration and provides a resolution path."
}
```


## **Observations**

Write your observations here

# **Output & Compilation**

In [173]:
# Write your code here
# Initialise storage lists - one entry per ticket, in the same order as the DataFrame
summaries = []   # Generated summaries
sum_eval  = []   # Parsed summarisation evaluation dicts
responses = []   # Generated customer responses
resp_eval = []   # Parsed response evaluation dicts

In [100]:
for idx, row in df.iterrows():
    ticket_id   = row["support_ticket_id"]
    description = row["support_ticket_desc"]

    # ── Step 1: Generate summary ──────────────────────────────────────────────
    summary = generate_summary(description)
    summaries.append(summary)

    # ── Step 2: Evaluate the summary ─────────────────────────────────────────
    sum_eval.append(evaluate_summary(description, summary))

    # ── Step 3: Generate customer response ───────────────────────────────────
    response = generate_response(summary)
    responses.append(response)

    # ── Step 4: Evaluate the response ────────────────────────────────────────
    resp_eval.append(evaluate_response(summary, response))

    print(f"  Ticket {ticket_id} - done")

print(f"\nPipeline complete. Processed {len(df)} tickets.")
print(f"Summaries: {len(summaries)}, Responses: {len(responses)}, Summary evals: {len(sum_eval)}, Response evals: {len(resp_eval)}")

  Ticket 1 - done
  Ticket 2 - done
  Ticket 3 - done
  Ticket 4 - done
  Ticket 5 - done
  Ticket 6 - done
  Ticket 7 - done
  Ticket 8 - done
  Ticket 9 - done
  Ticket 10 - done
  Ticket 11 - done
  Ticket 12 - done
  Ticket 13 - done
  Ticket 14 - done
  Ticket 15 - done
  Ticket 16 - done
  Ticket 17 - done
  Ticket 18 - done
  Ticket 19 - done
  Ticket 20 - done
  Ticket 21 - done
  Ticket 22 - done
  Ticket 23 - done
  Ticket 24 - done
  Ticket 25 - done
  Ticket 26 - done
  Ticket 27 - done
  Ticket 28 - done
  Ticket 29 - done
  Ticket 30 - done

Pipeline complete. Processed 30 tickets.
Summaries: 30, Responses: 30, Summary evals: 30, Response evals: 30


## **Combine All Outputs into a Single Consolidated Table**

In [101]:
# Write your code here
output_df = pd.DataFrame({
    "support_ticket_id"  : df["support_ticket_id"].values,
    "support_ticket_desc": df["support_ticket_desc"].values,
    "generated_summary"  : summaries,
    "generated_response" : responses
})

In [175]:
display(output_df.head(5))
print("Output dataframe shape:", output_df.shape)

,support_ticket_id,support_ticket_desc,generated_summary,generated_response
0,1,I cannot believe the level of service I have r...,Customer is frustrated with support service an...,"Dear [Customer's Name],\n\nThank you for reach..."
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri...",Customer's order SNX-8902 failed due to ACH de...,Subject: Assistance with Order SNX-8902\n\nDea...
2,3,not working. please help.,The customer reports an unspecified issue and ...,"Dear Customer,\n\nThank you for reaching out t..."
3,4,Okay so I have genuinely had it with this comp...,Customer is frustrated with repeated order iss...,"Dear [Customer's Name],\n\nThank you for reach..."
4,5,refund not received,Customer reports not receiving a refund.,"Dear Customer,\n\nThank you for reaching out t..."


Output dataframe shape: (30, 4)


In [ ]:
output_path = "C:\\datascience\\SupportTicketAnalysis\\output.csv"  
output_df.to_csv(output_path, index=False)                       # Save the file

print(f"Output saved to       : {output_path}")

Output saved to       : C:\datascience\SupportTicketAnalysis\output.csv


### Save the Evaluation Scores for Summarization and Response Generation

In [104]:
summary_evaluations = []
for eval_item in sum_eval:
    if isinstance(eval_item, str):
        try:
            eval_data = json.loads(eval_item)
        except json.JSONDecodeError:
            eval_data = {}
    else:
        eval_data = eval_item or {}

    scores = eval_data.get("scores", {})
    overall_verdict = eval_data.get("overall_verdict")

    evaluation_entry = {
        "technical_accuracy": scores.get("technical_accuracy"),
        "completeness": scores.get("completeness"),
        "conciseness": scores.get("conciseness"),
        "hallucination_check": scores.get("hallucination_check"),
        "summary_overall_verdict": overall_verdict,
    }
    summary_evaluations.append(evaluation_entry)

summary_eval_df = pd.DataFrame(summary_evaluations)

response_evaluations = []
for eval_item in resp_eval:
    if isinstance(eval_item, str):
        try:
            eval_data = json.loads(eval_item)
        except json.JSONDecodeError:
            eval_data = {}
    else:
        eval_data = eval_item or {}

    scores = eval_data.get("scores", {})
    feedback = eval_data.get("feedback", {})
    overall_verdict = eval_data.get("overall_verdict")

    evaluation_entry = {
        "alignment_with_summary": scores.get("alignment_with_summary"),
        "actionability": scores.get("actionability"),
        "tone_empathy": scores.get("tone_empathy"),
        "policy_compliance": scores.get("policy_compliance"),
        "response_overall_verdict": overall_verdict,
        "feedback_strengths": str(feedback.get("strengths")),
        "feedback_weaknesses": str(feedback.get("weaknesses")),
        "feedback_risk_factors": str(feedback.get("risk_factors")),
    }
    response_evaluations.append(evaluation_entry)

response_eval_df = pd.DataFrame(response_evaluations)

print(f"Parsed summary evaluations: {len(summary_eval_df)}, response evaluations: {len(response_eval_df)}")

Parsed summary evaluations: 30, response evaluations: 30


In [106]:
display(summary_eval_df.head())
summary_eval_df.to_csv("C:\\datascience\\SupportTicketAnalysis\\summary_evaluation.csv", index=False)   

,technical_accuracy,completeness,conciseness,hallucination_check,summary_overall_verdict
0,NaN,NaN,NaN,NaN,None
1,NaN,NaN,NaN,NaN,None
2,NaN,NaN,NaN,NaN,None
3,NaN,NaN,NaN,NaN,None
4,NaN,NaN,NaN,NaN,None


In [107]:
display(response_eval_df.head())
response_eval_df.to_csv( "C:\\datascience\\SupportTicketAnalysis\\response_evaluation.csv", index=False)   

,alignment_with_summary,actionability,tone_empathy,policy_compliance,response_overall_verdict,feedback_strengths,feedback_weaknesses,feedback_risk_factors
0,NaN,NaN,NaN,NaN,None,None,None,None
1,1.0,1.0,1.0,1.0,The response is effectively aligned with the c...,The response is well-aligned with the ticket s...,The response lacks specific details on how to ...,There is a risk of confusion if the payment pr...
2,NaN,NaN,NaN,NaN,None,None,None,None
3,NaN,NaN,NaN,NaN,None,None,None,None
4,NaN,NaN,NaN,NaN,None,None,None,None


# **Business Insights & Recommendations**

## **Business Insights**

Write your insights here

## **Recommendations**

Write your recommendations here

<font size=6>Power Ahead!</font>
___